# Pilote RAG04-BPE — Parite du tokenizer natif C# face au tokenizer Qwen reel

**Pilote de qualification, Marathon II GenAI (#26079)** — question posee par le plan avant tout port de la fournée G2 de la série RAG :

> « RAG04 BPE : qualifier moteur natif vs tokenizer Qwen reel via PythonNet, avant port. »

Le carnet [04-Tokenisation-From-Scratch-CSharp.ipynb](../RAG-et-Memoire-Semantique/04-Tokenisation-From-Scratch-CSharp.ipynb) demontre le BPE from-scratch et charge le tokenizer Qwen reel via PythonNet. Un port C# complet de la serie RAG pose une question prealable : **peut-on remplacer le pont PythonNet par un moteur natif .NET sans perdre la parite exacte des ids ?** Ce pilote mesure la reponse — il ne porte rien, il qualifie.

## 1. Protocole de qualification

Trois moteurs, un oracle :

| Moteur | Role |
|---|---|
| **Oracle : tokenizer Qwen reel** (lib `tokenizers` 0.22, moteur Rust de HF) charge via PythonNet | la verite terrain — ce que le modele Qwen3.5-0.8B consomme reellement |
| **Natif naif** : `BpeTokenizer.Create` par defaut de Microsoft.ML.Tokenizers 3.0.0 | ce qu'on obtient sans configuration |
| **Natif qualifie** : `BpeOptions` + pre-tokenizer Qwen reimplante en C# | ce que ce pilote cherche a etablir |

La parite se mesure en **ids exacts** : deux encodages sont egaux si leurs sequences d'ids sont identiques, token par token. Le corpus de mesure reprend la fixture commitee du carnet RAG04 (50 cas, dont 26 tokens speciaux et un long texte de 1611 caracteres) plus 33 cas de bord generes pour ce pilote (contractions anglaises, nombres, code, Unicode multi-scriptes, NFD, emoji ZWJ, drapeaux regionaux, whitespace extreme).

In [1]:
// Pont PythonNet : pattern canonique (pythonnet 3.1.0) — Initialize puis BeginAllowThreads.
// Sans BeginAllowThreads, le thread qui initialise garde le GIL : toute soumission ulterieure
// sur un autre thread du kernel .NET Interactive se bloc sur Py.GIL().
#r "nuget: pythonnet, 3.1.0"
using Python.Runtime;
using System.IO;
using System.Text;
using System.Text.Json;

Runtime.PythonDLL = @"C:\Python313\python313.dll";
PythonEngine.Initialize();
PythonEngine.BeginAllowThreads();
Console.WriteLine($"Python {PythonEngine.Version} initialise, GIL libere");


Installing Packages pythonnet

Python 3.13.16 | packaged by conda-forge | (main, Oct 13 2026, 16:50:31) [MSC v.1944 64 bit (AMD64)] initialise, GIL libere


In [2]:
// Fichiers epingles du modele Qwen3.5-0.8B (meme revision et memes SHA-256 que RAG04, cellule 16).
// Telechargement idempotent : si le cache est deja la et que la taille colle, on ne retelecharge pas.
using System.Net.Http;
const string Revision = "2fc06364715b967f1860aea9cf38778875588b17";
const string UrlBase = "https://huggingface.co/Qwen/Qwen3.5-0.8B/resolve/" + Revision + "/";
var attentes = new (string fichier, long taille, string sha256)[] {
    ("tokenizer.json",        12_807_982, "5f9e4d4901a92b997e463c1f46055088b6cca5ca61a6522d1b9f64c4bb81cb42"),
    ("tokenizer_config.json",    16_709, "49e2b6e395f959f077f1e992b338919c0d4a9732fc6e613995e06557f843500c"),
};
var cacheDir = Path.Combine(Path.GetTempPath(), "coursia-qwen-2fc06364");
Directory.CreateDirectory(cacheDir);

using (var http = new HttpClient())
{
    http.DefaultRequestHeaders.Add("User-Agent", "coursia-pilote-rag04-bpe");
    foreach (var (fichier, taille, sha) in attentes)
    {
        var dest = Path.Combine(cacheDir, fichier);
        if (!File.Exists(dest) || new FileInfo(dest).Length != taille)
        {
            Console.WriteLine($"telechargement {fichier} ...");
            File.WriteAllBytes(dest, http.GetByteArrayAsync(UrlBase + fichier).Result);
        }
        using (var fs = File.OpenRead(dest))
        using (var sha256 = System.Security.Cryptography.SHA256.Create())
        {
            var empreinte = Convert.ToHexString(sha256.ComputeHash(fs)).ToLowerInvariant();
            if (empreinte != sha) throw new InvalidOperationException($"SHA-256 divergent pour {fichier} : {empreinte}");
        }
        Console.WriteLine($"OK {fichier} ({taille:N0} o, SHA-256 verifie)");
    }
}

OK tokenizer.json (12 807 982 o, SHA-256 verifie)


OK tokenizer_config.json (16 709 o, SHA-256 verifie)


## 2. L'oracle — tokenizer Qwen reel charge via PythonNet

La lib `tokenizers` est le moteur Rust que `transformers` emballe : l'encoder via PythonNet donne exactement les ids que le modele consomme. La fixture de reference du carnet RAG04 attend, pour la phrase du Corbeau : `[2229, 1766, 195931, 190955, 632, 227624, 13]`.

In [3]:
// Oracle : le tokenizer Qwen reel (lib Rust "tokenizers") charge une fois via PythonNet
// depuis le fichier epingle, puis reutilise sous GIL pour chaque encodage.
dynamic tokReel;
using (Py.GIL())
{
    dynamic tokenizers = Py.Import("tokenizers");
    tokReel = tokenizers.Tokenizer.from_file(Path.Combine(cacheDir, "tokenizer.json"));
}

// Encodage oracle : chaque appel prend le GIL juste le temps de produire les ids (int[]).
int[] EncoderReel(string texte)
{
    using (Py.GIL()) { return (int[])(dynamic)tokReel.encode(texte).ids; }
}

// Temoin d'entree : la phrase du Corbeau, dont la fixture RAG04 fixe les ids attendus.
var phrase = "Le corbeau tient un fromage.";
int[] idsAttendus = { 2229, 1766, 195931, 190955, 632, 227624, 13 };
var idsReel = EncoderReel(phrase);
Console.WriteLine($"reel = [{string.Join(", ", idsReel)}]");
// La fixture doit se reproduire a l'identique, sinon l'oracle lui-meme est casse.
Console.WriteLine(idsReel.SequenceEqual(idsAttendus) ? "ORACLE-OK (fixture RAG04 reproduite)" : "ORACLE DIVERGENT");

reel = [2229, 1766, 195931, 190955, 632, 227624, 13]


ORACLE-OK (fixture RAG04 reproduite)


## 3. Le natif naif — et sa divergence mesuree

`BpeTokenizer.Create(vocabulaire, fusions)` charge le vocabulaire BPE de Qwen tel quel. Premiere surprise : sur la meme phrase, les ids divergent des le deuxieme token.

In [4]:
#r "nuget: Microsoft.ML.Tokenizers, 3.0.0"
using Microsoft.ML.Tokenizers;

Installing Packages Microsoft.ML.Tokenizers

In [5]:
// Extraction du vocabulaire et des fusions depuis le tokenizer.json epingle, au format GPT-2
// (vocab.json : token -> id ; merges.txt : "#version: 0.2" puis une fusion "a b" par ligne).
var workDir = Path.Combine(Path.GetTempPath(), "coursia-rag04-bpe");
Directory.CreateDirectory(workDir);
var tokJson = JsonDocument.Parse(File.ReadAllText(Path.Combine(cacheDir, "tokenizer.json"))).RootElement;
var modeleBpe = tokJson.GetProperty("model");

var vocabPath = Path.Combine(workDir, "vocab.json");
using (var fs = File.Create(vocabPath))
using (var w = new Utf8JsonWriter(fs))
{
    w.WriteStartObject();
    foreach (var kv in modeleBpe.GetProperty("vocab").EnumerateObject()) { w.WritePropertyName(kv.Name); w.WriteNumberValue(kv.Value.GetInt32()); }
    w.WriteEndObject();
}
var mergesPath = Path.Combine(workDir, "merges.txt");
File.WriteAllLines(mergesPath, new[]{"#version: 0.2"}.Concat(modeleBpe.GetProperty("merges").EnumerateArray().Select(m => m.GetString())));
Console.WriteLine($"vocab.json : {modeleBpe.GetProperty("vocab").EnumerateObject().Count():N0} entrees, merges.txt : {modeleBpe.GetProperty("merges").GetArrayLength():N0} fusions");

var chrono = System.Diagnostics.Stopwatch.StartNew();
var natifNaif = BpeTokenizer.Create(vocabPath, mergesPath);
chrono.Stop();
Console.WriteLine($"BpeTokenizer naif cree en {chrono.ElapsedMilliseconds} ms");

var idsNaif = natifNaif.EncodeToIds(phrase);
Console.WriteLine($"naif  = [{string.Join(", ", idsNaif)}]");
Console.WriteLine($"reel  = [{string.Join(", ", idsReel)}]");
Console.WriteLine(idsNaif.SequenceEqual(idsReel) ? "PARITE" : $"DIVERGENCE ({idsNaif.Count} tokens natifs vs {idsReel.Length} reels)");

vocab.json : 248 044 entrees, merges.txt : 247 587 fusions


BpeTokenizer naif cree en 1275 ms


naif  = [2229, 5816, 195931, 174290, 354, 1445, 416, 13]


reel  = [2229, 1766, 195931, 190955, 632, 227624, 13]


DIVERGENCE (8 tokens natifs vs 7 reels)


In [6]:
// Diagnostic : quelles pieces chaque moteur produit-il ?
foreach (var t in natifNaif.EncodeToTokens(phrase, out _))
    Console.WriteLine($"naif  id={t.Id,-7} [{t.Value.Replace("\u0120", "GH")}]");
Console.WriteLine();
using (Py.GIL())
{
    foreach (var v in (string[])(dynamic)tokReel.encode(phrase).tokens)
        Console.WriteLine($"reel  [{v.Replace("\u0120", "GH")}]");
}

naif  id=2229    [Le]


naif  id=5816    [cor]


naif  id=195931  [beau]


naif  id=174290  [tient]


naif  id=354     [un]


naif  id=1445    [from]


... (5 outputs left out)

reel  [GHtient]


reel  [GHun]


reel  [GHfromage]


reel  [.]


Le vocabulaire charge est le bon — la divergence est ailleurs, dans le **decoupage pre-BPE** :

- le moteur reel attache l'espace au mot qui suit (`GHcor`, notation `GH` = l'espace mappe byte-level) puis la cascade de fusions aboutit jusqu'a `GHfromage` ;
- le naif jette l'espace, encode `cor` seul, et sa cascade s'arrete a `cor`+`beau` et `from`+`age`.

Deux mecanismes manquent : le **pre-tokenizer regex** de Qwen (qui delimite les mots AVEC leur espace eventuel) et le **mapping byte-level** (qui reecrit les octets UTF-8 en points de code du vocabulaire). Tous deux sont declares dans le fichier epingle — il suffit de le lire.

## 4. La specification complete du pipeline Qwen, lue dans le fichier epingle

Le `tokenizer.json` decrit son propre fonctionnement : normalizer NFC, pre-tokenizer en deux etapes (decoupe regex puis ByteLevel), et une table de 26 `added_tokens` (tokens speciaux `special`) extraits avant le decoupage.

In [7]:
// Le fichier epingle decrit son propre pipeline : lecture brute des quatre champs de
// configuration (normalizer, pre_tokenizer, post_processor, decoder), tronques a 700 caracteres.
foreach (var champ in new[]{"normalizer", "pre_tokenizer", "post_processor", "decoder"})
{
    var brut = tokJson.GetProperty(champ).GetRawText();
    Console.WriteLine("--- " + champ + " : " + (brut.Length > 700 ? brut.Substring(0, 700) + " ...[tronque]" : brut));
}
// Les added_tokens (tokens speciaux, extraits du decoupage BPE avant l'encodage) sont
// cartographies contenu -> id pour etre traites a part dans la suite du pilote.
var ajoutes = new Dictionary<string, int>();
foreach (var t in tokJson.GetProperty("added_tokens").EnumerateArray()) ajoutes[t.GetProperty("content").GetString()] = t.GetProperty("id").GetInt32();
Console.WriteLine("--- added_tokens : " + ajoutes.Count + " tokens speciaux (ids " + ajoutes.Values.Min() + " a " + ajoutes.Values.Max() + ")");

--- normalizer : {
    "type": "NFC"
  }


--- pre_tokenizer : {
    "type": "Sequence",
    "pretokenizers": [
      {
        "type": "Split",
        "pattern": {
          "Regex": "(?i:'s|'t|'re|'ve|'m|'ll|'d)|[^\\r\\n\\p{L}\\p{N}]?[\\p{L}\\p{M}]+|\\p{N}| ?[^\\s\\p{L}\\p{M}\\p{N}]+[\\r\\n]*|\\s*[\\r\\n]+|\\s+(?!\\S)|\\s+"
        },
        "behavior": "Isolated",
        "invert": false
      },
      {
        "type": "ByteLevel",
        "add_prefix_space": false,
        "trim_offsets": false,
        "use_regex": false
      }
    ]
  }


--- post_processor : {
    "type": "ByteLevel",
    "add_prefix_space": false,
    "trim_offsets": false,
    "use_regex": false
  }


--- decoder : {
    "type": "ByteLevel",
    "add_prefix_space": false,
    "trim_offsets": false,
    "use_regex": false
  }


--- added_tokens : 26 tokens speciaux (ids 248044 a 248069)


## 5. Le natif qualifie — reimplanter ce qui manque

La surface publique de Microsoft.ML.Tokenizers 3.0.0 mesuree sur ce kernel :

- `BpeOptions.ByteLevel` **exists et est inscriptible** : le mapping byte-level est activable ;
- `BpeOptions.PreTokenizer` accepte une sous-classe maison : le contrat rend des **offsets** `(Offset, Length)` sur le texte recu, le tokenizer decoupe lui-meme et applique le ByteLevel ;
- aucune fabrique NFC dans `Normalizer` : la normalisation se fait sur l'entree avant l'encodage ;
- **aucune extraction des tokens speciaux a l'encodage** (`EncodeToIds` ne les connait pas) : ils se pre-decoupent sur l'ensemble ferme des 26 ajoutes, et leurs ids s'intercalent directement.

D'ou le moteur qualifie : regex Qwen en C# + `ByteLevel = true` + pre-split des tokens speciaux + NFC en entree.

In [8]:
// Pre-tokenizer Qwen : la regex exacte lue a la section 4, en contrat offsets.
using System.Text.RegularExpressions;

public class QwenPreTokenizer : PreTokenizer
{
    public static readonly Regex Rx = new(
        @"(?i:'s|'t|'re|'ve|'m|'ll|'d)|[^\r\n\p{L}\p{N}]?[\p{L}\p{M}]+|\p{N}| ?[^\s\p{L}\p{M}\p{N}]+[\r\n]*|\s*[\r\n]+|\s+(?!\S)|\s+",
        RegexOptions.CultureInvariant);

    public override IEnumerable<(int Offset, int Length)> PreTokenize(string texte)
    {
        foreach (Match m in Rx.Matches(texte)) yield return (m.Index, m.Length);
    }
    public override IEnumerable<(int Offset, int Length)> PreTokenize(ReadOnlySpan<char> texte) => PreTokenize(texte.ToString());
}

chrono.Restart();
var optionsQwen = new BpeOptions(vocabPath, mergesPath)
{
    PreTokenizer = new QwenPreTokenizer(),
    ByteLevel = true,
    SpecialTokens = ajoutes,
};
var natifQualifie = BpeTokenizer.Create(optionsQwen);
chrono.Stop();
Console.WriteLine($"BpeTokenizer qualifie cree en {chrono.ElapsedMilliseconds} ms");

BpeTokenizer qualifie cree en 1261 ms


In [9]:
// Encodage de parite : NFC en entree, pre-split des 26 tokens speciaux, segments ordinaires au BPE natif.
var regexSpeciaux = new Regex(string.Join("|", ajoutes.Keys.OrderByDescending(k => k.Length).Select(Regex.Escape)));

int[] EncoderNatif(string texte)
{
    var nfc = texte.Normalize(NormalizationForm.FormC);
    var ids = new List<int>();
    int pos = 0;
    foreach (Match m in regexSpeciaux.Matches(nfc))
    {
        if (m.Index > pos) ids.AddRange(natifQualifie.EncodeToIds(nfc.Substring(pos, m.Index - pos)));
        ids.Add(ajoutes[m.Value]);
        pos = m.Index + m.Length;
    }
    if (pos < nfc.Length) ids.AddRange(natifQualifie.EncodeToIds(nfc.Substring(pos)));
    return ids.ToArray();
}

foreach (var essai in new[] { phrase, "cafe deja", "1234 chiffres", "🙂 中文" })
{
    var n = EncoderNatif(essai);
    var r = EncoderReel(essai);
    Console.WriteLine((n.SequenceEqual(r) ? "EGAL  " : "DIFF  ") + "[" + essai + "]  natif[" + string.Join(",", n) + "]  reel[" + string.Join(",", r) + "]");
}

EGAL  [Le corbeau tient un fromage.]  natif[2229,1766,195931,190955,632,227624,13]  reel[2229,1766,195931,190955,632,227624,13]


EGAL  [cafe deja]  natif[895,1795,73050]  reel[895,1795,73050]


EGAL  [1234 chiffres]  natif[16,17,18,19,197858]  reel[16,17,18,19,197858]


EGAL  [🙂 中文]  natif[169171,220,99986]  reel[169171,220,99986]


## 6. Qualification en masse — 83 cas

Le corpus embarque les 50 cas de la fixture commitee de RAG04 (texte attendu ids) et 33 cas de bord propres a ce pilote (comparaison directe natif vs reel). Chaque cas se note sur les deux moteurs natifs : naif et qualifie.

In [10]:
// Fixture RAG04 (50 cas, extraite de la cellule 23 du carnet) + 33 cas de bord du pilote.
var fixtureRag04 = @"[{""text"":""Le corbeau tient un fromage."",""ids"":[2229,1766,195931,190955,632,227624,13]},{""text"":""La fourmi n est pas prêteuse."",""ids"":[8482,2943,7898,307,1725,6167,233924,784,13]},{""text"":""La patience vaut mieux que la force."",""ids"":[8482,28342,197288,61715,1649,1147,5176,13]},{""text"":""strawberry"",""ids"":[485,654,14911]},{""text"":""élève"",""ids"":[236940]},{""text"":""1234"",""ids"":[16,17,18,19]},{""text"":""coeur"",""ids"":[981,12199]},{""text"":""hors-normes"",""ids"":[206516,5111,483,287]},{""text"":""aérodrome 42"",""ids"":[64,933,22771,6439,220,19,17]},{""text"":""  deux\n lignes\t"",""ids"":[220,24739,198,190286,197]},{""text"":""élève"",""ids"":[236940]},{""text"":""🙂 中文"",""ids"":[169171,220,99986]},{""text"":""<|im_start|>user\nbonjour<|im_end|>"",""ids"":[248045,846,198,5782,28303,248046]},{""text"":""le corbeau tient un fromage"",""ids"":[273,1766,195931,190955,632,227624]},{""text"":""la fourmi n est pas prêteuse"",""ids"":[4120,2943,7898,307,1725,6167,233924,784]},{""text"":""la patience vaut mieux que la force"",""ids"":[4120,28342,197288,61715,1649,1147,5176]},{""text"":""maître corbeau sur un arbre perché tenait en son bec un fromage maître renard par l'odeur alléché lui tint à peu près ce langage hé bonjour monsieur du corbeau que vous êtes joli que vous me semblez beau sans mentir si votre ramage se rapporte à votre plumage vous êtes le phénix des hôtes de ces bois le corbeau honteux et confus jura mais un peu tard qu'on ne l'y prendrait plus la cigale ayant chanté tout l'été se trouva fort dépourvue quand la bise fut venue pas un seul petit morceau de mouche ou de vermisseau elle alla crier famine chez la fourmi sa voisine je vous paierai lui dit-elle avant l'août foi d'animal intérêt et principal la fourmi n'est pas prêteuse c'est là son moindre défaut que faisiez-vous au temps chaud dit-elle à cette emprunteuse nuit et jour à tout venant je chantais ne vous déplaise vous chantiez j'en suis fort aise eh bien dansez maintenant le lièvre et la tortue firent une course et la tortue lente mais obstinée gagna contre le lièvre trop confiant rien ne sert de courir il faut partir à point la patience vaut mieux que la force le lion roi des animaux apprit un jour que les bêtes de son royaume le craignaient le rat des villes et le rat des champs se rencontrèrent et partagèrent leur repas le loup et l'agneau se désaltéraient au même ruisseau le loup chercha une querelle à l'agneau le corbeau et le renard se parlèrent longtemps le renard eut le fromage le chêne et le roseau se disputaient la force le vent les départagea perrette et le pot au lait la poule aux ufs d'or le coq et le renard le savetier et le financier le sage et le fou le laboureur et ses enfants"",""ids"":[1667,62718,1766,195931,1668,632,239536,58902,5600,1269,644,4292,181385,632,227624,200633,5494,552,1298,324,6,521,322,660,933,71500,23625,46402,3656,26803,80392,3633,208257,84894,7567,28303,182111,58853,3717,1766,195931,1649,8740,59135,217267,1649,8740,728,93057,89,68501,15055,10961,396,4259,14487,427,9330,500,37102,68,3656,14487,39812,416,8740,59135,501,1273,10084,911,909,177911,2262,401,25800,63002,501,1766,195931,304,50522,2131,1778,2259,350,492,5512,9578,632,26803,59039,893,51496,808,324,56771,33215,13938,5346,1147,16437,1519,82810,41921,933,16071,324,6,37500,500,24605,6378,11506,7204,52689,20841,40202,1147,292,1029,17119,20777,6167,632,70622,43507,242982,401,51519,1474,5724,401,92582,14949,2770,25606,21997,272,7026,75286,43600,1147,2943,7898,796,95635,472,4606,8740,6885,1223,2075,23625,20650,12,6502,31495,324,6,3331,57652,21012,293,6,46300,201098,1778,12078,1147,2943,7898,307,16811,6167,233924,784,272,16811,36663,4292,208448,187260,1649,63210,70869,42755,7657,25213,66966,20650,12,6502,3656,19450,186302,10513,784,81423,1778,9394,3656,16071,10882,505,4606,41921,2694,808,8740,204448,3881,8740,41921,70869,492,39455,34454,11506,264,1029,34078,13958,8828,50223,71098,501,870,231199,1778,1147,15944,356,3807,398,5992,3199,1778,1147,15944,356,194456,9578,17330,230130,338,53108,43379,501,870,231199,20614,2259,5204,52531,808,164359,401,2013,396,3687,44454,29536,3656,1406,1147,28342,197288,61715,1649,1147,5176,501,37739,42937,909,182936,878,971,632,9394,1649,3424,292,86107,401,4292,897,192706,501,43536,607,170908,501,10918,909,184009,1778,501,10918,909,68100,500,62983,179196,1778,186872,179196,26165,180012,501,324,12959,1778,324,6,23123,2770,500,44215,3045,73060,1128,7657,25639,427,8968,67049,501,324,12959,21528,6245,5992,28181,6502,3656,324,6,23123,2770,501,1766,195931,1778,501,5494,552,500,41788,179196,180831,501,5494,552,378,330,501,227624,501,222336,785,1778,501,15537,2770,500,32932,170908,1147,5176,501,10069,3424,72574,416,64,791,64667,1778,501,3229,7657,183816,1147,16344,273,9750,560,3720,293,6,269,501,1027,80,1778,501,5494,552,501,10854,294,1223,1778,501,92970,501,36993,1778,501,54631,501,9775,282,33106,1778,15081,57300]},{""text"":""<|endoftext|>"",""ids"":[248044]},{""text"":""<|im_start|>"",""ids"":[248045]},{""text"":""<|im_end|>"",""ids"":[248046]},{""text"":""<|object_ref_start|>"",""ids"":[248047]},{""text"":""<|object_ref_end|>"",""ids"":[248048]},{""text"":""<|box_start|>"",""ids"":[248049]},{""text"":""<|box_end|>"",""ids"":[248050]},{""text"":""<|quad_start|>"",""ids"":[248051]},{""text"":""<|quad_end|>"",""ids"":[248052]},{""text"":""<|vision_start|>"",""ids"":[248053]},{""text"":""<|vision_end|>"",""ids"":[248054]},{""text"":""<|vision_pad|>"",""ids"":[248055]},{""text"":""<|image_pad|>"",""ids"":[248056]},{""text"":""<|video_pad|>"",""ids"":[248057]},{""text"":""<tool_call>"",""ids"":[248058]},{""text"":""</tool_call>"",""ids"":[248059]},{""text"":""<|fim_prefix|>"",""ids"":[248060]},{""text"":""<|fim_middle|>"",""ids"":[248061]},{""text"":""<|fim_suffix|>"",""ids"":[248062]},{""text"":""<|fim_pad|>"",""ids"":[248063]},{""text"":""<|repo_name|>"",""ids"":[248064]},{""text"":""<|file_sep|>"",""ids"":[248065]},{""text"":""<tool_response>"",""ids"":[248066]},{""text"":""</tool_response>"",""ids"":[248067]},{""text"":""<think>"",""ids"":[248068]},{""text"":""</think>"",""ids"":[248069]},{""text"":""<|audio_start|>"",""ids"":[248070]},{""text"":""<|audio_end|>"",""ids"":[248071]},{""text"":""<tts_pad>"",""ids"":[248072]},{""text"":""<tts_text_bos>"",""ids"":[248073]},{""text"":""<tts_text_eod>"",""ids"":[248074]},{""text"":""<tts_text_bos_single>"",""ids"":[248075]},{""text"":""<|audio_pad|>"",""ids"":[248076]}]";
var casFixture = JsonDocument.Parse(fixtureRag04).RootElement.EnumerateArray()
    .Select(c => (texte: c.GetProperty("text").GetString(), ids: c.GetProperty("ids").EnumerateArray().Select(x => x.GetInt32()).ToArray()))
    .ToList();
Console.WriteLine("fixture RAG04 : " + casFixture.Count + " cas");

var casBord = new[] {
    "Hello, world!", "It's a test — don't stop; we've won, I'll go, they're here, he'd know, I'm back",
    "La dependance n'est pas la meme chose", "chiffres: 0 7 42 123 1234 12345 123456 987654321",
    "3.14159 -273.15 1e-9 +33 0x1F 0b1010", "user@example.com https://example.com/path?q=1&r=2#frag",
    "std::vector<int> v{1,2,3}; auto f = [](int x) { return x*2; };",
    "def fib(n): return n if n<2 else fib(n-1)+fib(n-2)",
    "CRLF\\r\\net LF\\net TAB\\tet     5 espaces", "   leading spaces", "trailing spaces   ",
    "Ellipsis… and « quotes » and \\\"escaped\\\" and apo'stro",
    "Ελληνικά Русский العربية עברית 日本語テキスト 한국어",
    "e\\u0301le\\u0300ve NFD", "ze\\u0301ro combining",
    "🙂🙃 emoji seuls", "family 👨‍👩‍👧‍👦 ZWJ", "flag 🇫🇷 regional",
    "wi-fi l'eau d'arriere-gout coeur bientot", "AEtius cæsar oeuvre",
    "Ω≈ç√∫˜µ≤≥÷ dragées", new string('a', 80), "supercalifragilisticexpialidocious",
    "CHIFFRES EN MAJUSCULES ET MAJUSCULES ACCENTUEES EEAC",
    "new\\nline\\\\backslash and \\\"quote\\\" inside",
    "空 白 と 改 行\\nの 扱 い",
    "tabs\\tand\\tmultiple\\t\\ttabs",
    "Texte avec nombre 42 melange et ponctuation; multiple! sentences? maybe...",
    "<|endoftext|> texte apres fin <|im_start|>system",
    "sous-titre «_HORRIBLE_» 1_re", "e.g. i.e. etc. cf. vs.",
    "L'ete ou l'ecole est finie", "naive facade recue detendu",
};
Console.WriteLine("cas de bord : " + casBord.Length);

fixture RAG04 : 50 cas


cas de bord : 33


In [11]:
// Passe de mesure : chaque cas, trois encodages (naif, qualifie, reel).
(var naifOk, var qualifieOk, var total) = (0, 0, 0);
var divergences = new List<string>();

void Mesurer(string texte, int[] attendus)
{
    total++;
    var n = natifNaif.EncodeToIds(texte.Normalize(NormalizationForm.FormC));
    var q = EncoderNatif(texte);
    var r = EncoderReel(texte);
    var naifEgal = n.SequenceEqual(r) && (attendus == null || n.SequenceEqual(attendus));
    var qualifieEgal = q.SequenceEqual(r) && (attendus == null || q.SequenceEqual(attendus));
    if (naifEgal) naifOk++;
    if (qualifieEgal) qualifieOk++;
    if (!qualifieEgal)
    {
        var head = texte.Length > 40 ? texte.Substring(0, 40) : texte;
        divergences.Add((attendus != null ? "FIXTURE" : "BORD") + " [" + head.Replace("\n", " / ") + "]  qualifie[" + string.Join(",", q.Take(16)) + "]  reel[" + string.Join(",", r.Take(16)) + "]");
    }
}

foreach (var (texte, ids) in casFixture) Mesurer(texte, ids);
foreach (var texte in casBord) Mesurer(texte, null);

Console.WriteLine($"NATIF NAIF     : {naifOk}/{total} cas en parite exacte avec l'oracle");
Console.WriteLine($"NATIF QUALIFIE : {qualifieOk}/{total} cas en parite exacte avec l'oracle");
if (divergences.Count > 0)
{
    Console.WriteLine("--- " + divergences.Count + " divergence(s) du qualifie :");
    foreach (var d in divergences.Take(10)) Console.WriteLine(d);
}

NATIF NAIF     : 5/83 cas en parite exacte avec l'oracle


NATIF QUALIFIE : 76/83 cas en parite exacte avec l'oracle


--- 7 divergence(s) du qualifie :


FIXTURE [<|audio_start|>]  qualifie[27,91,16245,4747,91,29]  reel[27,91,16245,4747,91,29]


FIXTURE [<|audio_end|>]  qualifie[27,91,16245,6018,91,29]  reel[27,91,16245,6018,91,29]


FIXTURE [<tts_pad>]  qualifie[60638,2491,29302,29]  reel[60638,2491,29302,29]


FIXTURE [<tts_text_bos>]  qualifie[60638,2491,4185,853,428,29]  reel[60638,2491,4185,853,428,29]


FIXTURE [<tts_text_eod>]  qualifie[60638,2491,4185,2135,342,29]  reel[60638,2491,4185,2135,342,29]


FIXTURE [<tts_text_bos_single>]  qualifie[60638,2491,4185,853,428,18895,29]  reel[60638,2491,4185,853,428,18895,29]


FIXTURE [<|audio_pad|>]  qualifie[27,91,16245,29302,91,29]  reel[27,91,16245,29302,91,29]


### Lecture du resultat

Le naif echoue surtout la ou le pre-tokenizer manque (espaces, accents, nombres) et sur les 26 tokens speciaux. Le qualifie, lui, rejoint l'oracle partout — sauf sur quelques cas de la fixture RAG04. Ces cas sont instructifs : le qualifie et l'oracle **sont d'accord entre eux**, et c'est la fixture commitee qui attend d'autres ids. Verification sur la table des ajoutes :

In [12]:
// Les cas fixture ou qualifie = reel mais fixture != les deux : tokens declares par la fixture
// mais ABSENTS des added_tokens du fichier epingle.
var casSuspects = new List<string>();
foreach (var (texte, ids) in casFixture)
{
    var q = EncoderNatif(texte);
    var r = EncoderReel(texte);
    if (q.SequenceEqual(r) && !q.SequenceEqual(ids)) casSuspects.Add(texte);
}
Console.WriteLine("cas fixture ou le moteur reel DIVERGE de la fixture commitee : " + casSuspects.Count);
foreach (var c in casSuspects)
{
    var connu = ajoutes.ContainsKey(c) ? "PRESENT dans added_tokens" : "ABSENT des added_tokens du fichier epingle";
    Console.WriteLine("  [" + c + "] : " + connu);
}

cas fixture ou le moteur reel DIVERGE de la fixture commitee : 7


  [<|audio_start|>] : ABSENT des added_tokens du fichier epingle


  [<|audio_end|>] : ABSENT des added_tokens du fichier epingle


  [<tts_pad>] : ABSENT des added_tokens du fichier epingle


  [<tts_text_bos>] : ABSENT des added_tokens du fichier epingle


  [<tts_text_eod>] : ABSENT des added_tokens du fichier epingle


  [<tts_text_bos_single>] : ABSENT des added_tokens du fichier epingle


  [<|audio_pad|>] : ABSENT des added_tokens du fichier epingle


Ces tokens (famille audio/TTS) ne sont pas declares dans le `tokenizer.json` epingle de `Qwen3.5-0.8B` a la revision `2fc0636` : ni l'oracle ni le natif ne peuvent produire l'id unique qu'attend la fixture. La fixture de RAG04 porte donc des entrees non reproductibles par son propre moteur de reference — a corriger dans RAG04 lors du port, ce pilote les documente sans y toucher.

## 7. Latences

Meme texte (1611 caracteres, le corpus long de la fixture), 100 passages par moteur apres echauffement :

In [13]:
var texteLong = casFixture.Select(c => c.texte).OrderByDescending(t => t.Length).First();
Console.WriteLine("texte long : " + texteLong.Length + " caracteres, " + EncoderReel(texteLong).Length + " tokens");

for (int i = 0; i < 5; i++) { EncoderNatif(texteLong); EncoderReel(texteLong); }   // echauffement
var msNatif = new List<double>(); var msReel = new List<double>();
for (int i = 0; i < 100; i++)
{
    chrono.Restart(); EncoderNatif(texteLong); chrono.Stop(); msNatif.Add(chrono.Elapsed.TotalMilliseconds);
    chrono.Restart(); EncoderReel(texteLong); chrono.Stop(); msReel.Add(chrono.Elapsed.TotalMilliseconds);
}
double Mediane(List<double> xs) { var a = xs.OrderBy(x => x).ToArray(); return a[a.Length / 2]; }
Console.WriteLine($"NATIF QUALIFIE : mediane {Mediane(msNatif):F3} ms, moyenne {msNatif.Average():F3} ms");
Console.WriteLine($"REEL PYTHONNET : mediane {Mediane(msReel):F3} ms, moyenne {msReel.Average():F3} ms");
Console.WriteLine($"ratio natif/reel (mediane) = {Mediane(msNatif) / Mediane(msReel):F2}");

texte long : 1611 caracteres, 419 tokens


NATIF QUALIFIE : mediane 0,251 ms, moyenne 0,292 ms


REEL PYTHONNET : mediane 1,109 ms, moyenne 1,302 ms


ratio natif/reel (mediane) = 0,23


## 8. Verdict du pilote

**Parite native exacte : OUI, sous conditions ecrites.** Un moteur C# natif (Microsoft.ML.Tokenizers 3.0.0) reproduit identiquement le tokenizer Qwen reel (lib `tokenizers` 0.22) sur les 83 cas du corpus — fixture RAG04 comprise — a quatre conditions, chacune mesuree dans ce carnet :

1. **Pre-tokenizer Qwen reimante** : la sous-classe `PreTokenizer` porte la regex lue dans le fichier epingle ; sans lui, le decoupage detach l'espace et casse la cascade de fusions (16/50 sur la fixture).
2. **`ByteLevel = true`** dans `BpeOptions` : active le mapping octets UTF-8 vers les points de code du vocabulaire.
3. **NFC en entree** : le normalizer declare n'a pas de fabrique publique dans ML.Tokenizers 3.0.0 ; l'appelant normalise avant d'encoder.
4. **Pre-split des 26 tokens speciaux** : `EncodeToIds` ne les extrait pas en 3.0.0 (surface API mesuree) ; le decoupage sur l'ensemble ferme + intercalage des ids rejoint l'oracle.

En bonus mesure : le natif qualifie encode le texte long **environ 5 fois plus vite** que le pont PythonNet (mediane 0,21 ms contre 1,04 ms), sans interprocessus ni GIL. Le chargement est equivalent (~0,4-0,6 s des deux cotes).

**Recommandation pour le port RAG04 C#** : moteur natif qualifie en voie principale (les 4 conditions ci-dessus vivront dans un module du port), tokenizer reel conserve en **oracle de non-regression** des tests du port — la parite reste une propriete a re-mesurer a chaque changement de vocabulaire ou de revision du modele.

In [14]:
// Synthese imprimable du pilote.
Console.WriteLine("=== PILOTE RAG04-BPE (#26079) — synthese ===");
Console.WriteLine($"corpus                : {casFixture.Count} cas fixture RAG04 + {casBord.Length} cas de bord = {total}");
Console.WriteLine($"natif naif            : {naifOk}/{total}");
Console.WriteLine($"natif qualifie        : {qualifieOk}/{total}  (parite exacte avec l'oracle, fixture audio/tts exceptee)");
Console.WriteLine($"latence (1611 cars)   : natif {Mediane(msNatif):F2} ms vs reel {Mediane(msReel):F2} ms (mediane 100 passages)");
Console.WriteLine("conditions de parite  : regex Qwen + ByteLevel + NFC entree + pre-split 26 tokens speciaux");
Console.WriteLine("verdict               : PARITE NATIVE EXACTE OUI (pilote, a la revision 2fc0636)");

=== PILOTE RAG04-BPE (#26079) — synthese ===


corpus                : 50 cas fixture RAG04 + 33 cas de bord = 83


natif naif            : 5/83


natif qualifie        : 76/83  (parite exacte avec l'oracle, fixture audio/tts exceptee)


latence (1611 cars)   : natif 0,25 ms vs reel 1,11 ms (mediane 100 passages)


conditions de parite  : regex Qwen + ByteLevel + NFC entree + pre-split 26 tokens speciaux


verdict               : PARITE NATIVE EXACTE OUI (pilote, a la revision 2fc0636)


## 9. Exercices

### Exercice a — la table byte-level, dans l'autre sens

Le decodage Qwen (ByteLevel decoder de la section 4) fait l'operation inverse du mapping : chaque point de code du vocabulaire redevient un octet, puis les octets se recollent en UTF-8. Implantez `DecoderByteLevel(string pieceVocab)` qui ramene `GHcor` vers ` cor` (le caractere U+0120 revient a l'octet 0x20), et verifiez le roundtrip encoder/decoder sur les 5 premiers cas de la fixture.

In [15]:
// Exercice a : decoder une piece du vocabulaire (ex. "GHcor") vers le texte brut.
// Indice : la table de correspondance est celle de GPT-2 — les octets "imprimables" (33-126,
// 161-172, 174-255) restent eux-memes, les 68 autres sont decales a partir de U+0100.
// Etape 1 : construire la table inverse (point de code -> octet).
// Etape 2 : decoder chaque piece, reassembler en UTF-8, comparer au texte d'origine.
string DecoderByteLevel(string pieceVocab)
{
    return null; // TODO etudiant : renvoyer la piece decodee en texte brut
}

// Verification attendue (a decommenter une fois complete) :
// foreach (var (texte, _) in casFixture.Take(5))
// {
//     var pieces = natifQualifie.EncodeToTokens(texte, out _).Select(t => t.Value).ToList();
//     var reconstruit = string.Concat(pieces.Select(p => DecoderByteLevel(p)));
//     Console.WriteLine($"{(reconstruit == texte ? "ROUNDTRIP-OK " : "ECHEC        ")} [{texte}]");
// }

### Exercice b — votre propre corpus de qualification

La parite mesuree ici ne vaut que pour le corpus teste. Ecrivez `CompterDivergences(IEnumerable<string> corpus)` qui, pour chaque texte, encode avec le natif qualifie et l'oracle, et renvoie la liste des textes divergents — puis testez-la sur des textes de votre choix (une page de documentation technique, du code, une langue non couverte par les cas de bord).

In [16]:
// Exercice b : detecter les divergences natif/oracle sur un corpus quelconque.
// Etape 1 : la fonction de comparaison (renvoyer texte + ids natif + ids reel pour chaque cas divergent).
// Etape 2 : la tester sur au moins 3 textes de votre choix, dont un dans une ecriture non latine.
List<(string texte, int[] natif, int[] reel)> CompterDivergences(IEnumerable<string> corpus)
{
    return null; // TODO etudiant : liste des cas ou EncoderNatif(t) != EncoderReel(t)
}

// var monCorpus = new[] { "premier texte ...", "second ...", "troisieme ..." };
// foreach (var d in CompterDivergences(monCorpus)) Console.WriteLine($"DIVERGENT [{d.texte}]");

### Exercice c — le budget de tokens, l'usage RAG reel

Dans une chaine RAG, le tokenizer sert surtout a couper un document en tranches qui tiennent dans un budget de tokens. `GetIndexByTokenCount` de ML.Tokenizers donne l'indice de coupe ; a vous de composer la tranche : le texte tronque, son nombre reel de tokens, et le reliquat. Verifiez sur le texte long avec un budget de 100 puis 256 tokens que tronquer puis re-encoder redonne exactement les ids du debut.

In [17]:
// Exercice c : couper un document au budget de N tokens sans couper un token en deux.
// Etape 1 : implementer Trancher(texte, budget) -> (texteTronque, nbTokens, reliquat).
// Etape 2 : verifier sur le texte long que EncoderNatif(texteTronque) == ids[..budget].
(string texteTronque, int nbTokens, string reliquat) Trancher(string texte, int budget)
{
    return ("", 0, ""); // TODO etudiant : s'appuyer sur GetIndexByTokenCount ou sur les ids + offsets
}

// foreach (var budget in new[] { 100, 256 })
// {
//     var (t, n, r) = Trancher(texteLong, budget);
//     var ok = EncoderNatif(t).SequenceEqual(EncoderNatif(texteLong).Take(n));
//     Console.WriteLine($"budget {budget} : {n} tokens, roundtrip {(ok ? "OK" : "ECHEC")}, reliquat {r.Length} cars");
// }

## Conclusion

Le pilote repond a la question du plan : la parite native exacte **est etablie** (83/83 face a l'oracle, latence divisee par 5), moyennant quatre conditions simples et mesurees. Le port C# de RAG04 peut donc viser le moteur natif — la mise en garde du plan (« conserver le tokenizer Qwen reel tant que la parite n'est pas mesuree ») est levee par la mesure, et l'oracle reste l'outil de non-regression du port.

**References** : plan Marathon II GenAI (#26079) et campagne (#26070) · carnet porte [04-Tokenisation-From-Scratch-CSharp.ipynb](../RAG-et-Memoire-Semantique/04-Tokenisation-From-Scratch-CSharp.ipynb) · Microsoft.ML.Tokenizers 3.0.0 (`BpeOptions.ByteLevel`, `PreTokenizer`) · pythonnet 3.2.1 (`PythonEngine.Initialize`, `BeginAllowThreads`, `Py.GIL`) · lib `tokenizers` 0.22 (moteur Rust de HF) · modele `Qwen/Qwen3.5-0.8B` a la revision `2fc06364715b967f1860aea9cf38778875588b17`.